## CIS 5800 Machine Perception 26 Spring
## Homework 2 - AR Pose Estimation

## Instructions
- This is the coding part for homework 2 and worth 50 points.
- Start early! Please post your questions on [Ed Discussion](https://edstem.org/us/courses/93331/discussion) or come to office hours!
- You can work on this using Google Colab (recommended) or running it locally.

## Submission

**All coding assignments must be submitted through Gradescope.**

You need to submit the following `.py` files:
- `est_Pw.py`
- `est_homography.py`
- `procrustes.py`
- `pnp.py`
- `p3p.py`
- `est_pixel_world.py`

Complete **all sections marked with `##### STUDENT CODE START #####`** in each file.

## Library Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt

## Load Student Code

In [2]:
from est_Pw import est_Pw
from est_homography import est_homography
from procrustes import Procrustes
from pnp import PnP
from p3p import P3P
from est_pixel_world import est_pixel_world

## Camera Intrinsics

The camera intrinsic matrix used throughout this assignment.

In [3]:
K = np.array([
    [823.8, 0.0, 304.8],
    [0.0, 822.8, 236.3],
    [0.0, 0.0, 1.0]
])
print("Camera Intrinsics K:\n", K)

Camera Intrinsics K:
 [[823.8   0.  304.8]
 [  0.  822.8 236.3]
 [  0.    0.    1. ]]


---
## 1. World Coordinates (`est_Pw`)

Estimate the world coordinates of the April tag corners. The world origin is at the center of the tag, and the xy plane lies in the plane of the tag. See `world_setup.jpg` for the corner ordering (a, b, c, d).

**Points: 5**

In [4]:
# Test est_Pw
s = 0.14  # April tag side length in meters
Pw = est_Pw(s)

print("Pw shape:", Pw.shape)
print("Pw:\n", Pw)

expected_Pw = np.array([
    [0, 0, 0],
    [0, s, 0],
    [s, s, 0],
    [s, 0, 0]
])

assert Pw.shape == (4, 3), f"Expected shape (4,3), got {Pw.shape}"
assert np.allclose(Pw, expected_Pw, atol=1e-6), "Values do not match expected corners"
print("est_Pw: PASSED")

Pw shape: (4, 3)
Pw:
 [[0.   0.   0.  ]
 [0.   0.14 0.  ]
 [0.14 0.14 0.  ]
 [0.14 0.   0.  ]]
est_Pw: PASSED


---
## 2. Homography Estimation (`est_homography`)

Compute the 3x3 homography matrix H such that Y ~ H*X, given 4 point correspondences.

**Points: 10**

In [5]:
# Test est_homography
X = np.array([[10, 10], [100, 10], [100, 100], [10, 100]], dtype=np.float64)
Y = np.array([[20, 30], [120, 20], [150, 130], [40, 140]], dtype=np.float64)
H = est_homography(X, Y)

# Verify H maps X -> Y
X_h = np.hstack((X, np.ones((4, 1))))
Y_pred_h = (H @ X_h.T).T
Y_pred = Y_pred_h[:, :2] / Y_pred_h[:, 2:3]

print("H:\n", H)
print("\nPredicted Y:\n", Y_pred)
print("Expected Y:\n", Y)
print("Max error:", np.max(np.abs(Y_pred - Y)))

assert H.shape == (3, 3), f"Expected shape (3,3), got {H.shape}"
assert np.allclose(Y_pred, Y, atol=0.1), "Homography does not map X to Y correctly"
print("\nest_homography: PASSED")

H:
 [[ 1.08865248e+00  1.80851064e-01  7.09219858e+00]
 [-1.11702128e-01  1.07269504e+00  2.00709220e+01]
 [-8.86524823e-05 -9.75177305e-04  1.00000000e+00]]

Predicted Y:
 [[ 20.  30.]
 [120.  20.]
 [150. 130.]
 [ 40. 140.]]
Expected Y:
 [[ 20.  30.]
 [120.  20.]
 [150. 130.]
 [ 40. 140.]]
Max error: 1.7820411812863313e-11

est_homography: PASSED


---
## 3. Procrustes (`Procrustes`)

Solve the Procrustes problem: given point sets X and Y where Y = R@X + t, recover the rotation R and translation t.

**Points: 10**

In [6]:
# Test Procrustes
theta = np.radians(30)
R_true = np.array([
    [np.cos(theta), -np.sin(theta), 0],
    [np.sin(theta),  np.cos(theta), 0],
    [0, 0, 1]
])
t_true = np.array([1.0, -0.5, 0.3])

np.random.seed(42)
X_proc = np.random.randn(10, 3)
Y_proc = (R_true @ X_proc.T).T + t_true

R_est, t_est = Procrustes(X_proc, Y_proc)

print("R_est:\n", R_est)
print("R_true:\n", R_true)
print("\nt_est:", t_est)
print("t_true:", t_true)

# Check valid rotation
assert np.allclose(R_est @ R_est.T, np.eye(3), atol=1e-4), "R is not orthogonal"
assert np.isclose(np.linalg.det(R_est), 1.0, atol=1e-4), "det(R) != 1"

# Check reconstruction
Y_recon = (R_est @ X_proc.T).T + t_est
assert np.allclose(Y_recon, Y_proc, atol=1e-4), "Y != R@X + t"
print("\nProcrustes: PASSED")

R_est:
 [[ 8.66025404e-01 -5.00000000e-01  1.08115193e-17]
 [ 5.00000000e-01  8.66025404e-01 -1.36084195e-17]
 [-9.95085540e-17  2.04562040e-16  1.00000000e+00]]
R_true:
 [[ 0.8660254 -0.5        0.       ]
 [ 0.5        0.8660254  0.       ]
 [ 0.         0.         1.       ]]

t_est: [ 1.  -0.5  0.3]
t_true: [ 1.  -0.5  0.3]

Procrustes: PASSED


---
## 4. Perspective-N-Point (`PnP`)

Solve the PnP problem using collineation (homography decomposition). Given 4 world-pixel correspondences and camera intrinsics K, recover the camera pose (R_wc, t_wc).

**Points: 10**

In [7]:
# Test PnP
theta_pnp = np.radians(15)
R_wc_true = np.array([
    [np.cos(theta_pnp), -np.sin(theta_pnp), 0],
    [np.sin(theta_pnp),  np.cos(theta_pnp), 0],
    [0, 0, 1]
])
t_wc_true = np.array([0.1, 0.05, 0.5])

s = 0.14
Pw_test = np.array([[0, 0, 0], [0, s, 0], [s, s, 0], [s, 0, 0]])

# Project world pts to pixels
R_cw_true = np.linalg.inv(R_wc_true)
t_cw_true = -R_cw_true @ t_wc_true
Pc_test = np.zeros((4, 2))
for i in range(4):
    p_cam = R_cw_true @ Pw_test[i] + t_cw_true
    p_pix = K @ p_cam
    Pc_test[i] = p_pix[:2] / p_pix[2]

R_est, t_est = PnP(Pc_test, Pw_test, K)

# Reprojection check
R_cw_est = np.linalg.inv(R_est)
t_cw_est = -R_cw_est @ t_est.flatten()
reproj_err = 0
for i in range(4):
    p_cam = R_cw_est @ Pw_test[i] + t_cw_est
    p_pix = K @ p_cam
    p_pix = p_pix[:2] / p_pix[2]
    reproj_err = max(reproj_err, np.max(np.abs(p_pix - Pc_test[i])))

print("Max reprojection error:", reproj_err, "px")
print("det(R):", np.linalg.det(R_est))

assert np.allclose(R_est @ R_est.T, np.eye(3), atol=1e-2), "R is not orthogonal"
assert np.isclose(np.linalg.det(R_est), 1.0, atol=1e-2), "det(R) != 1"
assert reproj_err < 2.0, f"Reprojection error too high: {reproj_err}"
print("\nPnP: PASSED")

Max reprojection error: 9.72022462519817e-11 px
det(R): 0.9999999999999984

PnP: PASSED


---
## 5. Perspective-3-Point (`P3P`)

Solve the P3P problem. Uses 3 world-pixel correspondences to compute candidate poses, then selects the best using a 4th point.

**Points: 10**

In [8]:
# Test P3P
theta_p3p = np.radians(20)
R_wc_true_p3p = np.array([
    [np.cos(theta_p3p), -np.sin(theta_p3p), 0],
    [np.sin(theta_p3p),  np.cos(theta_p3p), 0],
    [0, 0, 1]
])
t_wc_true_p3p = np.array([0.08, 0.04, 0.45])

s = 0.14
Pw_test_p3p = np.array([[0, 0, 0], [0, s, 0], [s, s, 0], [s, 0, 0]])

# Project world pts to pixels
R_cw_true_p3p = np.linalg.inv(R_wc_true_p3p)
t_cw_true_p3p = -R_cw_true_p3p @ t_wc_true_p3p
Pc_test_p3p = np.zeros((4, 2))
for i in range(4):
    p_cam = R_cw_true_p3p @ Pw_test_p3p[i] + t_cw_true_p3p
    p_pix = K @ p_cam
    Pc_test_p3p[i] = p_pix[:2] / p_pix[2]

R_est_p3p, t_est_p3p = P3P(Pc_test_p3p, Pw_test_p3p, K)

# Reprojection check
R_cw_est_p3p = np.linalg.inv(R_est_p3p)
t_cw_est_p3p = -R_cw_est_p3p @ t_est_p3p.flatten()
reproj_err_p3p = 0
for i in range(4):
    p_cam = R_cw_est_p3p @ Pw_test_p3p[i] + t_cw_est_p3p
    p_pix = K @ p_cam
    p_pix = p_pix[:2] / p_pix[2]
    reproj_err_p3p = max(reproj_err_p3p, np.max(np.abs(p_pix - Pc_test_p3p[i])))

print("Max reprojection error:", reproj_err_p3p, "px")
print("det(R):", np.linalg.det(R_est_p3p))

assert np.allclose(R_est_p3p @ R_est_p3p.T, np.eye(3), atol=1e-2), "R is not orthogonal"
assert np.isclose(np.linalg.det(R_est_p3p), 1.0, atol=1e-2), "det(R) != 1"
assert reproj_err_p3p < 2.0, f"Reprojection error too high: {reproj_err_p3p}"
print("\nP3P: PASSED")

Max reprojection error: 2.113971220296662e-10 px
det(R): 1.0000000000000004

P3P: PASSED


---
## 6. Pixel to World Back-Projection (`est_pixel_world`)

Given pixel coordinates, camera pose (R_wc, t_wc), and intrinsics K, back-project pixels to world coordinates on the z=0 plane.

**Points: 5**

In [9]:
# Test est_pixel_world
theta_epw = np.radians(10)
R_wc_epw = np.array([
    [np.cos(theta_epw), -np.sin(theta_epw), 0],
    [np.sin(theta_epw),  np.cos(theta_epw), 0],
    [0, 0, 1]
])
t_wc_epw = np.array([0.05, 0.03, 0.4])

# Known world points on z=0 plane
Pw_known = np.array([
    [0.0, 0.0, 0.0],
    [0.1, 0.0, 0.0],
    [0.1, 0.1, 0.0],
    [0.0, 0.1, 0.0]
])

# Project to pixels
R_cw_epw = np.linalg.inv(R_wc_epw)
t_cw_epw = -R_cw_epw @ t_wc_epw
pixels_epw = np.zeros((4, 2))
for i in range(4):
    p_cam = R_cw_epw @ Pw_known[i] + t_cw_epw
    p_pix = K @ p_cam
    pixels_epw[i] = p_pix[:2] / p_pix[2]

# Back-project
Pw_est = est_pixel_world(pixels_epw, R_wc_epw, t_wc_epw, K)

print("Estimated world coords:\n", Pw_est)
print("Expected world coords:\n", Pw_known)
print("Max error:", np.max(np.abs(Pw_est - Pw_known)))

assert Pw_est.shape == (4, 3), f"Expected shape (4,3), got {Pw_est.shape}"
assert np.allclose(Pw_est, Pw_known, atol=0.01), "Back-projection does not match"
print("\nest_pixel_world: PASSED")

Estimated world coords:
 [[-2.22044605e-17  2.22044605e-17  0.00000000e+00]
 [ 1.00000000e-01  4.44089210e-17  0.00000000e+00]
 [ 1.00000000e-01  1.00000000e-01  0.00000000e+00]
 [-2.22044605e-17  1.00000000e-01  0.00000000e+00]]
Expected world coords:
 [[0.  0.  0. ]
 [0.1 0.  0. ]
 [0.1 0.1 0. ]
 [0.  0.1 0. ]]
Max error: 4.4408920985006264e-17

est_pixel_world: PASSED


---
## Summary

| Function | Points |
|----------|--------|
| `est_Pw` | 5 |
| `est_homography` | 10 |
| `Procrustes` | 10 |
| `PnP` | 10 |
| `P3P` | 10 |
| `est_pixel_world` | 5 |
| **Total** | **50** |

If all cells above print `PASSED`, you should receive full marks on Gradescope.